In [13]:
import pandas as pd
import numpy as np

In [14]:
max_rows = 100000
np.random.seed(42)


In [15]:
# --- Helper function to generate plausible apartment area ---
def get_area_based_on_rooms(total_rooms):
    avg_area_per_room = {
        1: (30, 5),   # 20–40 m² typical
        2: (43, 8),   # 27–59 m²
        3: (57, 10),  # 27–87 m²
        4: (69, 12),  # 33–105 m²
        5: (80, 15)   # 35–125 m²
    }
    
    # For >5 rooms, add ~13 m² per extra room
    if total_rooms in avg_area_per_room:
        mean, std = avg_area_per_room[total_rooms]
    else:
        mean = 80 + (total_rooms - 5) * 13
        std = 15 + (total_rooms - 5) * 3

    # Draw normally distributed area
    area = np.random.normal(mean, std)
    # Clip area to realistic Danish range 20–150 m²
    area = np.clip(area, 20, 150)
    return round(area, 1)


In [16]:
# --- Helper function to generate rents based on sqm and rooms ---
def generate_rents(sqm, rooms):
    # price per sqm drawn randomly within real range
    price_per_sqm = np.random.uniform(40, 80, size=len(sqm))
    # room premium (per extra room above 2)
    room_premium = (rooms - 2) * np.random.normal(200, 50, size=len(sqm))
    # base rent
    rents = sqm * price_per_sqm + room_premium
    # add some noise for location/condition
    rents += np.random.normal(0, 1000, size=len(sqm))
    rents = np.clip(rents, 1000, 15000)
    return rents.round().astype(int)


In [17]:
# --- Helper function to generate children based on adults ---
def generate_children(adults):
    children = np.zeros_like(adults, dtype=int)
    
    # Case 1: Adults = 1
    mask_single = (adults == 1)
    has_kids_single = np.random.rand(mask_single.sum()) < np.random.uniform(0.10, 0.15)
    children[mask_single] = has_kids_single * np.random.poisson(1, mask_single.sum())
    
    # Case 2: Adults = 2
    mask_couple = (adults == 2)
    children[mask_couple] = np.random.poisson(1, mask_couple.sum())
    
    children = np.clip(children, 0, 5)
    return children



In [18]:
# --- Helper function to calculate number of rooms based on adults and children ---
def calculate_rooms(adults, children):
    # Base room count: if adults = 1 or 2 → 1 bedroom
    bedrooms = 1 if adults in (1, 2) else 0

    # Bedrooms for children
    if children == 0:
        bedrooms += 0
    elif children == 1:
        bedrooms += 1  # 1 extra bedroom for 1 child
    elif children == 2:
        bedrooms += np.random.choice([1, 2], p=[0.5, 0.5])  # higher chance of 2 rooms
    elif children >= 3:
        bedrooms += np.random.choice([2, 3], p=[0.2, 0.8])  # most likely 3

    # Living room (optional) - 90% chance to have one
    living_room = 1 if np.random.rand() < 0.9 else 0

    return bedrooms + living_room

In [19]:
# --- Data generation ---
ages = np.random.choice(range(15, 40), max_rows)
adults = np.random.choice([1,2], max_rows, p=[0.75,0.25])
#children = np.random.poisson(1, max_rows)                                 #--------------- 0-> 36%, 1-> 36%, 2-> 18%, 3-> 9%, 4-> 1%
children = generate_children(adults)

# TOTAL ROOMS
total_rooms = np.array([calculate_rooms(a, c) for a, c in zip(adults, children)])

# AREA OF APARTMENT
areas = np.array([get_area_based_on_rooms(r) for r in total_rooms])

rents = generate_rents(areas, total_rooms)

In [20]:

# Filtering for validity
valid_idx = (
    (rents > 0)
    & (adults >= 1)
    & (total_rooms >= 1)
    & (areas >= 10)  # plausible min area
    & (ages >= 18)  # plausible min age
)

# Trim arrays
ages = ages[valid_idx]
adults = adults[valid_idx]
children = children[valid_idx]
total_rooms = total_rooms[valid_idx]
areas = areas[valid_idx]
rents = rents[valid_idx]

# set children count to 0 for age less than 21.
children[ages <= 18] = 0

# Truncate to sample size/excel limit
sample_size = min(len(ages), len(adults), len(total_rooms), len(areas), len(rents), max_rows)
tenancy_distance = np.clip(np.random.normal(5, 4, sample_size), 0.5, None)

distance_to_university = np.clip(
    np.random.normal(2, 1, sample_size), 0.1, None
)
# Amenity distances, now not tied to area "type" anymore—could make these depend on area size if you want!
hospital_distance = np.clip(np.random.normal(6, 4, sample_size), 1.5, None)
gym_distance = np.clip(np.random.normal(2, 4, sample_size), 0.5, None)
school_distance = np.clip(np.random.normal(6, 4, sample_size), 2, None)
supermarket_distance = np.clip(np.random.normal(2, 2, sample_size), 0.5, None)

In [21]:

df = pd.DataFrame({
    'Age': ages[:sample_size],
    'Adults': adults[:sample_size],
    'Children': children[:sample_size],
    'Rent': rents[:sample_size],
    'Distance_to_New_Tenancy': tenancy_distance.round(2),
    'Total_Rooms': total_rooms[:sample_size],
    'Area_m2': areas[:sample_size],  # Now area in square meters
    'Hospital_distance': hospital_distance.round(2),
    'Gym_distance': gym_distance.round(2),
    'School_distance': school_distance.round(2),
    'Supermarket_distance': supermarket_distance.round(2),
    'Distance_to_University': np.round(distance_to_university, 2)
})

In [22]:
label = np.zeros(len(df))

In [23]:

# ------------------------------
# 1. Students - University priority, no school distance
label = (
    1
    - 0.06 * df['Distance_to_University'].fillna(0)   # highest weight
    - 0.06 * df['Rent'] / 1000
    - 0.04 * df['Supermarket_distance']
    - 0.03 * df['Gym_distance']
    + 0.005 * df['Total_Rooms']                       # reduced weight
)

# Normalize area (20–150 m² → [0,1]) for all rows
norm_area = (df['Area_m2'] - 20) / (150 - 20)
label += 0.05 * norm_area   # max contribution = 0.05
hosp = np.sqrt(df['Hospital_distance'].clip(lower=0))
label -= 0.03 * hosp

In [24]:
# ------------------------------
# Finalize
df['Label'] = np.clip(label, 0, 1).round(2)
# df['Label'] = label.round(2)

df.to_excel('synthetic_housing_data_for_student.xlsx', index=False)